# 04 · Modelo predictivo — entrenamiento simple y validación

Versión **didáctica** de la predicción de rendimiento agrícola (t/ha).

> El modelo de producción vive en `src/risk/predict_rendimiento.py` (XGBoost/RandomForest
> por cultivo, validación cruzada y explicabilidad SHAP). Aquí construimos una línea base
> simple para entender el flujo: features → split → entrenamiento → validación.


In [ ]:
# Arranque: localizar la raiz del repo y la base de datos DuckDB
import os, sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "config.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import duckdb
import numpy as np
import pandas as pd

from config import config

DB_PATH = ROOT / config.duckdb_path
DB_OK = DB_PATH.exists()
print("BD:", DB_PATH, "->", "OK" if DB_OK else "NO existe (corre `make pipeline`)")


def con():
    if not DB_OK:
        raise FileNotFoundError(f"No existe {DB_PATH}. Ejecuta el pipeline primero.")
    c = duckdb.connect(str(DB_PATH), read_only=True)
    try:
        c.execute("INSTALL spatial; LOAD spatial;")
    except Exception:
        pass
    return c


def q(sql: str) -> pd.DataFrame:
    with con() as c:
        return c.execute(sql).df()


In [ ]:
# Librerias de visualizacion
import matplotlib.pyplot as plt
try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except Exception:
    pass
plt.rcParams["figure.figsize"] = (10, 5)


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("scikit-learn cargado")


## 1. Dataset de entrenamiento

Objetivo (`y`): `rendimiento_promedio` (t/ha). Predictores (`X`): variables climáticas SPC
y de exposición SEP. Excluimos `rendimiento_cv` para evitar fuga de información desde el propio rendimiento.


In [ ]:
from src.features.normalize import _SPC_COLS, _SEP_COLS

fmc = q("SELECT * FROM features_municipio_cultivo")

target = "rendimiento_promedio"
predictores = [c for c in (_SPC_COLS + _SEP_COLS)
               if c in fmc.columns and c not in (target, "rendimiento_cv")]

datos = fmc.dropna(subset=[target]).copy()
datos = datos[datos[target] > 0]                      # rendimientos validos
datos[predictores] = datos[predictores].fillna(0.0)

X = datos[predictores]
y = datos[target]
print("Muestras:", len(X), "| Predictores:", len(predictores))
predictores


## 2. Split train/test y entrenamiento


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

modelo = RandomForestRegressor(
    n_estimators=200, max_depth=14, min_samples_leaf=5,
    n_jobs=-1, random_state=42
)
modelo.fit(X_train, y_train)
print("Modelo entrenado sobre", len(X_train), "muestras")


## 3. Validación básica


In [ ]:
pred = modelo.predict(X_test)

rmse = float(np.sqrt(mean_squared_error(y_test, pred)))   # compat. sklearn >=1.6 (sin squared=)
mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)

print(f"R2   : {r2:.3f}")
print(f"RMSE : {rmse:.3f} t/ha")
print(f"MAE  : {mae:.3f} t/ha")
print(f"Rendimiento medio real (test): {y_test.mean():.3f} t/ha")


In [ ]:
# Predicho vs real
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, pred, s=8, alpha=0.3, color="#00897b")
lim = [0, float(np.percentile(y_test, 99))]
ax.plot(lim, lim, "--", color="#c62828", label="ideal (y = x)")
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel("Rendimiento real (t/ha)")
ax.set_ylabel("Rendimiento predicho (t/ha)")
ax.set_title(f"Predicho vs real  (R2={r2:.2f})")
ax.legend()
plt.tight_layout(); plt.show()


## 4. Importancia de variables


In [ ]:
imp = (pd.Series(modelo.feature_importances_, index=predictores)
       .sort_values(ascending=False))

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(imp.index[::-1], imp.values[::-1], color="#5e35b1", edgecolor="white")
ax.set_title("Importancia de variables (RandomForest)")
ax.set_xlabel("importancia relativa")
plt.tight_layout(); plt.show()
imp.round(4).to_frame("importancia").head(10)


---
### Resumen

- Línea base simple: RandomForest predice el rendimiento (t/ha) a partir de variables climáticas y de exposición.
- Validación con split 80/20 y métricas R²/RMSE/MAE.
- La importancia de variables muestra qué factores pesan más en el rendimiento.

En producción esto se hace por cultivo, con validación cruzada, intervalos de confianza y
explicabilidad SHAP (`src/risk/predict_rendimiento.py` y `nnet_rendimiento.py`).

Siguiente paso → **`05_reportes_automaticos.ipynb`**.
